# Wrangling big data: when the data won't fit in memory

_Data Wrangling_

**Shrink dtypes, read in chunks, use Parquet, and reach for Polars or Dask before a bigger machine.**

Sooner or later a dataset gets big enough that loading it the naive way &mdash;
       pd.read_csv("everything.csv") &mdash; either crawls or crashes with a memory error.
       The instinct is to ask for a bigger machine. Usually you do not need one. The single most useful
       habit is: don't load what you don't need.

       This lesson is four cheap moves that buy you a lot of room before you reach for heavy machinery:

---

This notebook walks through those four moves one at a time — profile, shrink dtypes, chunk, and graduate to Parquet + Polars. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q polars
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — pandas + polars

### Step 1 — Profile: where do the bytes actually go?

Before optimizing anything, measure. We load the California housing data and tack on two deliberately wasteful columns: a text `price_band` stored as Python `object`, and a small-range `county_code` stored as `int64`. `df.info(memory_usage='deep')` and `memory_usage(deep=True)` show the real per-column cost (`deep=True` counts the actual string bytes, not just the pointer). Profiling tells you which columns are worth attacking.

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
df = data.frame.copy()

# Add a low-cardinality TEXT column, stored as object (the wasteful default).
price_bins = [0, 1, 2, 3, 4, 6]
price_labels = ['very_low', 'low', 'mid', 'high', 'very_high']
df['price_band'] = pd.cut(df['MedHouseVal'], bins=price_bins, labels=price_labels).astype(object)

# Add a small-range INTEGER column, stored as int64 (also wasteful).
rng = np.random.RandomState(0)
df['county_code'] = rng.randint(0, 58, size=len(df)).astype('int64')

# See dtypes + per-column memory; deep=True counts real string size.
df.info(memory_usage='deep')
print(df.memory_usage(deep=True))           # bytes per column
before_bytes = df.memory_usage(deep=True).sum()
print('before:', before_bytes)              # -> 2,920,574 bytes (~2.92 MB)

### Step 2 — Shrink dtypes: downcast numerics + categorize text

The cheapest win: store each value in the smallest type that holds it. We downcast `float64` columns to `float32` (2x smaller), downcast integers to the narrowest int that fits (here `int8`, 8x smaller), and convert the repetitive text column to `category` — which stores each distinct label once plus a tiny integer code per row (~60x on this column). No information is lost; only the storage shrinks.

In [ ]:
# float64 -> float32 (2x smaller per column).
for c in df.select_dtypes('float64').columns:
    df[c] = pd.to_numeric(df[c], downcast='float')

# int64 -> the narrowest int that fits (int8 here, 8x smaller).
for c in df.select_dtypes('integer').columns:
    df[c] = pd.to_numeric(df[c], downcast='integer')

# object text -> category (stores each label once + a small code per row).
df['price_band'] = df['price_band'].astype('category')

after_bytes = df.memory_usage(deep=True).sum()
print('after :', after_bytes)               # -> 784,932 bytes (~0.78 MB) => 3.7x

### Step 3 — Read in chunks: aggregate a file bigger than RAM

When a file is too big to load at once, stream it. `pd.read_csv(..., chunksize=...)` yields the file one block of rows at a time, so peak memory stays at roughly one chunk no matter how large the file is. `usecols` reads only the columns you need. We compute a per-chunk grouped sum and accumulate the partials into a running total — valid because summation is associative and order-independent. (This cell assumes a `huge_sales.csv` exists; it illustrates the pattern.)

In [ ]:
totals = None
for chunk in pd.read_csv('huge_sales.csv',
                         usecols=['region', 'amount'],   # read only needed columns
                         chunksize=1_000_000):           # 1M rows at a time
    part = chunk.groupby('region')['amount'].sum()
    totals = part if totals is None else totals.add(part, fill_value=0)

print(totals)   # exact global sum-by-region, never holding the whole file

### Step 4 — Graduate: Parquet + a Polars lazy scan

Parquet is a compressed, columnar format: it is smaller on disk, and because it is laid out by column you can read just the columns you need (column pruning). Polars' `scan_parquet` goes further — it is **lazy**, so a `.filter()` and column choice are pushed down *into* the read, and nothing is loaded until `.collect()`. For genuinely cluster-scale data, Dask offers the same column-pruned, pushed-down read across a cluster.

In [ ]:
# Parquet: compressed, columnar.
df.to_parquet('housing.parquet')

# Read only two columns; the others are never touched on disk.
sub = pd.read_parquet('housing.parquet', columns=['MedInc', 'price_band'])

# Polars lazy scan: filter + column choice pushed into the read.
result = (
    pl.scan_parquet('housing.parquet')      # lazy: nothing read yet
      .filter(pl.col('MedInc') > 5.0)        # predicate pushed down
      .group_by('price_band')
      .agg(pl.col('MedInc').mean().alias('avg_medinc'))
      .collect()                             # only now does it read the needed data
)
print(result)

# For genuinely cluster-scale data, the Dask analog is:
#   import dask.dataframe as dd
#   dd.read_parquet('housing.parquet')[['MedInc','price_band']].groupby('price_band').mean().compute()

## Visualize the data & results

_On a real mixed-dtype DataFrame, how much memory do you save by downcasting numerics (float64->float32, int64->int8) and converting a low-cardinality text column to 'category'?_

### Step 1 — Build a realistic mixed-dtype frame

To measure the savings concretely, we rebuild the same frame: the 20,640-row California housing data plus a low-cardinality text column (`price_band`, 5 labels stored as `object`) and a small-range integer column (`county_code` as `int64`). These are exactly the columns where pandas' default dtypes waste the most memory.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

# Real bundled dataset: 20,640 rows of California housing.
d = fetch_california_housing(as_frame=True)
df = d.frame.copy()

# Low-cardinality TEXT column (5 labels) stored as object.
price_bins = [0, 1, 2, 3, 4, 6]
price_labels = ['very_low', 'low', 'mid', 'high', 'very_high']
df['price_band'] = pd.cut(df['MedHouseVal'], bins=price_bins, labels=price_labels).astype(object)

# Small-range INTEGER column stored as int64.
rng = np.random.RandomState(0)
df['county_code'] = rng.randint(0, 58, size=len(df)).astype('int64')

### Step 2 — Measure memory before, by dtype group

With `memory_usage(deep=True)` we record the real byte cost of the float columns, the integer column, and the text column separately, plus the total. Grouping by dtype makes it clear afterward exactly where the savings came from.

In [ ]:
# deep=True counts real string size, not just the object pointer.
mb = df.memory_usage(deep=True)
floats_b = sum(mb[c] for c in df.select_dtypes('float64').columns)
int_b    = mb['county_code']
text_b   = mb['price_band']
total_b  = mb.sum()

### Step 3 — Downcast, then compare before vs after

Now apply the same three optimizations on a copy: downcast floats to `float32`, the integer to `int8`, and the text to `category`. We re-measure each dtype group and print before/after side by side. The text column shrinks the most (near-duplicate strings collapse to codes), and the total comes down about 3.7x.

In [ ]:
# Downcast numerics + convert text to category, on a copy.
df2 = df.copy()
for c in df2.select_dtypes('float64').columns:
    df2[c] = pd.to_numeric(df2[c], downcast='float')      # float64 -> float32
df2['county_code'] = pd.to_numeric(df2['county_code'], downcast='integer')  # -> int8
df2['price_band']  = df2['price_band'].astype('category')  # object -> category

# Re-measure each dtype group after optimization.
mb2 = df2.memory_usage(deep=True)
floats_a = sum(mb2[c] for c in df2.select_dtypes('float32').columns)
int_a    = mb2['county_code']
text_a   = mb2['price_band']
total_a  = mb2.sum()

rows = [('floats', floats_b, floats_a), ('int', int_b, int_a),
        ('text', text_b, text_a), ('TOTAL', total_b, total_a)]
for name, b, a in rows:
    print(f'{name:7s} before {b/1e6:6.2f} MB  after {a/1e6:6.2f} MB')
# floats  before   1.49 MB  after   0.74 MB
# int     before   0.17 MB  after   0.02 MB
# text    before   1.27 MB  after   0.02 MB
# TOTAL   before   2.92 MB  after   0.78 MB   (3.7x smaller)
print('reduction:', round(total_b / total_a, 2), 'x')   # -> 3.72 x

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your df.info(memory_usage="deep") shows a 4 GB frame, and one object column named country (only ~200 distinct values across 50 million rows) accounts for most of it. What single change helps most, and roughly why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice the column is low-cardinality text: about 200 distinct values, but 50 million rows. — _As object, pandas stores a full Python string per row, so 50M near-duplicate strings dominate the memory._
- Convert it with df["country"] = df["country"].astype("category"). — _category stores each of the ~200 distinct strings once and keeps a tiny integer code per row._
- Re-check memory_usage(deep=True) to confirm the drop. — _With 200 codes the per-row cost falls from a string to about one byte, cutting that column dramatically._

**Answer:** Convert country to the category dtype. Because there are only ~200 distinct values over 50M rows, pandas stores each label once plus a one- or two-byte code per row instead of a whole Python string per row &mdash; a many-x cut on the column that dominates the frame. Downcasting numerics helps too, but the repetitive text column is where the biggest single win is.

</details>

**Problem 2.** A teammate needs the total amount summed by region from a 60 GB CSV on a 16 GB-RAM laptop. pd.read_csv(path) crashes. Outline a chunked approach, and name one operation this approach would not support.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Iterate with for chunk in pd.read_csv(path, usecols=["region","amount"], chunksize=1_000_000):. — _Only two columns and one million rows are in memory at a time, so RAM stays flat regardless of file size._
- Per chunk, compute chunk.groupby("region")["amount"].sum() and add it into a running total Series. — _A grouped sum is associative: partial sums per chunk combine correctly into the global sum._
- After the loop, the accumulated Series is the answer. — _Summing is order-independent, so combining partials gives the exact global result._

**Answer:** Read in chunks with chunksize and usecols=["region","amount"], do a per-chunk groupby(...).sum(), and accumulate the partial sums into a running total. Memory stays at roughly one chunk. What this would not support is any operation that needs all the data at once &mdash; a global sort, an exact median, or a cross-file dedup &mdash; because those cannot be combined from independent per-chunk results. For those, reach for Polars, Dask, or a database.

</details>

**Problem 3.** You repeatedly read 3 of 80 columns from a dataset for different analyses, and each read is slow. The data is currently a single big CSV. What format change helps, and what two mechanisms make it faster?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Convert the CSV to Parquet once: pd.read_csv(path).to_parquet("data.parquet") (chunked if it does not fit). — _Parquet is a compressed columnar format, so it is smaller on disk and laid out by column._
- Read only what you need: pd.read_parquet("data.parquet", columns=["a","b","c"]). — _Columnar storage lets the reader physically skip the bytes of the other 77 columns (column pruning)._
- Add a filter and let the engine skip non-matching row blocks (predicate pushdown), especially via Polars' scan_parquet().filter(). — _Parquet stores per-block statistics, so blocks that cannot match the filter are never read._

**Answer:** Store the data as Parquet instead of CSV. Two mechanisms make repeated partial reads fast: column pruning &mdash; because Parquet is columnar, reading columns=["a","b","c"] physically skips the other 77 columns &mdash; and predicate pushdown &mdash; per-block statistics let the reader skip whole row groups that cannot match a filter. Compression also shrinks the file. A CSV, being row-major text, must scan the whole file every time.

</details>